In [7]:
from Association_Rule_Mining import normalize_values, perform_PCA, perform_LASSO, filter_apriori, create_tweet_dataset, create_continous_dataset, create_categorize_dataset, perform_apriori, create_triple_barrier_labeling
import pandas as pd

In [8]:
market_set = ['BTC-USD', 'AMZN', 'MSFT']
STARTING_DATE = "2018-01-01"
ENDING_DATE ="2020-01-01"

In [9]:
def create_dataset(market):
    continous_df = create_continous_dataset(market, starting_date=STARTING_DATE, ending_date=ENDING_DATE)
    continous_df_with_tbl = create_triple_barrier_labeling(continous_df)
    tweets_df = create_tweet_dataset("Datasets/investing_classified_sentiments" if market != "BTC-USD" else "Datasets/btc_classified_sentiments") #"Datasets/investing_classified_sentiments"
    merged_df = pd.concat([continous_df_with_tbl, tweets_df], axis=1).dropna()
    extended = merged_df #extend_time_columns(merged_df, skip_cols=["signals"], t=7)
    return extended

def get_antecedents_for_trade_profitable(df):
    pd.set_option('display.max_colwidth', None)
    all = filter_apriori(perform_apriori(df, min_support = 0.01, min_confidence=0.1), min_lift=0.1, target_label = 'trade_profitable_1', min_antecedents=3, max_antecedents=5)
    return all.drop_duplicates(subset=['antecedents'])

def filter_continous_columns(merged_df):
    continous_df = merged_df.copy()
    for col in merged_df.columns:
        if merged_df[col].nunique() <= 3:
            continous_df = continous_df.drop(col, axis=1)
    continous_df = continous_df.dropna()
    return continous_df

In [10]:
for market in market_set:
    print(f"Processing market: {market} *****")
    merged_df = create_dataset(market)
    categorized_for_trade_profitable = create_categorize_dataset(merged_df, suffix_vals=['bearish', 'Bearish', 'bullish', 'Bullish', '1.0', '1', '0.0', '0', '-1', '-1.0'], skip_cols=["next_day_label", "signals", "previous_label"])
    print(f"Categorized columns: {categorized_for_trade_profitable.columns}")
    print(get_antecedents_for_trade_profitable(categorized_for_trade_profitable))
    continous_df = filter_continous_columns(merged_df)
    print(f"Continous columns: {continous_df.columns}")
    X_scaled = normalize_values(continous_df)
    perform_PCA(X_scaled, y=X_scaled["close_pct_change"], top_k = 8)
    perform_LASSO(X_scaled, continous_df, target="close", n = 10)

[*********************100%***********************]  1 of 1 completed

Processing market: BTC-USD *****


Optimizing:   0%|          | 0/5 [00:00<?, ?it/s]

Optimizing:   0%|          | 0/5 [00:00<?, ?it/s]

Optimizing:   0%|          | 0/5 [00:00<?, ?it/s]

...

                                                                             antecedents  \
0        (crossover_0, sen_negative_increasing_0, close_increasing_1, MACD_Increasing_0)   
1       (RSI_bearish, sen_positive_increasing_1, sen_negative_increasing_1, crossover_0)   
2                (volatility_label_bullish, MACD_Increasing_1, ROC_bearish, crossover_1)   
3        (volatility_label_bullish, sen_positive_increasing_1, ROC_bearish, crossover_1)   
4         (MACD_Increasing_1, close_increasing_0, volatility_label_bullish, crossover_1)   
...                                                                                  ...   
2070                         (volatility_label_bullish, close_increasing_0, crossover_0)   
2071  (crossover_0, close_increasing_0, volatility_label_bullish, Momentum_Increasing_0)   
2072                               (sen_compound_increasing_1, RSI_bearish, crossover_0)   
2073        (sen_compound_increasing_1, RSI_bearish, crossover_0, Momentum_Incre

Optimizing:   0%|          | 0/5 [00:00<?, ?it/s]

Optimizing:   0%|          | 0/5 [00:00<?, ?it/s]

Optimizing:   0%|          | 0/5 [00:00<?, ?it/s]

Label distribution: 
2.0    153
0.0    115
1.0     93
Name: label, dtype: int64 0    1.576178
1    5.435264
2    4.483813
Name: sharpe_ratio, dtype: float64
Categorized columns: Index(['trade_profitable_0', 'trade_profitable_1', 'close_increasing_0',
       'close_increasing_1', 'crossover_0', 'crossover_1',
       'Momentum_Increasing_0', 'MACD_Increasing_0', 'MACD_Increasing_1',
       'volatility_label_bearish', 'volatility_label_bullish', 'RSI_bearish',
       'RSI_bullish', 'ROC_bearish', 'ROC_bullish',
       'sen_compound_increasing_0', 'sen_compound_increasing_1',
       'sen_positive_increasing_0', 'sen_positive_increasing_1',
       'sen_negative_increasing_0', 'sen_negative_increasing_1'],
      dtype='object')
                                                                                                      antecedents  \
0                                (sen_compound_increasing_1, volatility_label_bullish, sen_negative_increasing_1)   
1                   (sen_compound_